In [0]:
# %md
# ### Broze layer

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import re

In [0]:
processing_timestamp = F.current_timestamp()
print("Processing timestamp generated for data lineage tracking")


Processing timestamp generated for data lineage tracking


In [0]:
bronze_customers_initial_df = spark.createDataFrame(
    [
        (101, "  Acme Corp  ", "manufacturing", "north america", "Enterprise", True, "john.doe@acme.com", "+1-555-0101", "123 Main St", "New York", "NY", "10001", "2020-01-15", "A"),
        (102, "BRIGHT retail", "Retail", "North America", "mid-market", True, "contact@brightretail.com", "555.0102", "456 Oak Ave", "Chicago", "IL", "60601", "2019-03-22", "A"),
        (103, "City Hospital", "healthcare", "NORTH AMERICA", "Enterprise", True, "admin@cityhospital.org", "(555) 0103", "789 Health Blvd", "Boston", "MA", "02101", "2021-07-10", "A"),
        (104, "delta airlines", "Transportation", "north america", "Strategic", True, "info@delta.com", "1-555-0104", "1001 Airport Way", "Atlanta", "GA", "30301", "2018-11-05", "A"),
        (105, "Frontier Education ", "Education", "North America", "Mid-Market", True, "hello@frontier.edu", "555-0105", "2020 Campus Dr", "Austin", "TX", "73301", "2022-02-28", "A"),
        (106, "Gamma Stores", "retail", "latin america", "SMB", True, "ventas@gamma.com", "+52-555-0106", "Av. Principal 100", "Mexico City", "DF", "01000", "2020-09-12", "B"),
        (107, "HELIOS ENERGY", "Energy", "europe", "Strategic", True, "contact@helios.eu", "+44-20-5550107", "10 Energy Plaza", "London", "", "SW1A 1AA", "2019-06-18", "A"),
        (108, "innova Labs", "technology", "Europe", "Mid-Market", True, "info@innova.de", "+49-30-5550108", "Tech Str. 42", "Berlin", "BE", "10115", "2021-12-03", "A"),
    ],
    ["customerId", "customerName", "industry", "region", "customerTier", "isActive", "email", "phone", "address", "city", "state", "postalCode", "onboardDate", "creditRating"],
)

In [0]:
bronze_products_initial_df = spark.createDataFrame(
    [
        (1001, "  photon laptop 14  ", "computing", "865.50", "1299.00", "ACTIVE", "2023-01-15", "LT", "15.6", "Intel i7"),
        (1002, "LUMEN monitor 27", "Accessories", "205.0", "329", "active", "2023-02-20", "MN", "27.0", "4K Display"),
        (1003, "Nimbus Router PRO", "networking", "102.50", "189.99", "Active", "2023-03-10", "RT", "", "WiFi 6"),
        (1004, "aurora tablet 11", "Computing", "455", "749.0", "ACTIVE", "2023-04-05", "TB", "11.0", "ARM Processor"),
        (1005, "SolarDock pro", "accessories", "140.25", "229.00", "discontinued", "2022-12-01", "DK", "0", "USB-C Hub"),
        (1006, "ION SERVER blade", "Computing", "682.75", "1125", "Active", "2023-05-18", "SV", "1U", "Xeon Gold"),
    ],
    ["productId", "productName", "category", "unitCost", "listPrice", "status", "launchDate", "productCode", "screenSize", "processor"],
)

In [0]:
bronze_orders_initial_df = spark.createDataFrame(
    [
        (7001, 1, "2024-01-10", "2024-01-12", 101, 1001, "4", "1299.0", "usd", "SHIPPED", "field_sales", "REP001", "10.5", "Standard"),
        (7001, 2, "2024-01-10", "2024-01-13", 101, 1002, "2", "329.0", "USD", "shipped", "field_sales", "REP001", "5.0", "standard"),
        (7002, 1, "2024-01-12", "2024-01-14", 102, 1002, "3", "329", "usd", "Shipped", "partner", "PART01", "15.0", "Express"),
        (7003, 1, "2024-01-14", "2024-01-16", 103, 1003, "5", "189.99", "USD", "shipped", "online", "", "0", "Standard"),
        (7004, 1, "2024-01-18", "2024-01-20", 104, 1004, "1", "749.0", "USD", "SHIPPED", "online", "WEB001", "25.5", "express"),
        (7005, 1, "2024-01-20", "2024-01-24", 105, 1001, "2", "1299", "usd", "shipped", "field_sales", "REP002", "12.0", "Standard"),
        (7006, 1, "2024-01-22", None, 106, 1004, "3", "749.0", "USD", "processing", "online", "WEB002", "", "Standard"),
        (7007, 1, "2024-01-25", "2024-01-28", 107, 1005, "6", "229.00", "EUR", "shipped", "partner", "PART02", "8.5", "Standard"),
        (7008, 1, "2024-01-27", None, 108, 1006, "2", "1125.0", "USD", "PENDING", "field_sales", "REP003", "20.0", "Express"),
        (7009, 1, "2024-01-29", "2024-02-01", 105, 1003, "4", "189.99", "usd", "shipped", "online", "WEB001", "7.5", "standard"),
        (7010, 1, "2024-02-02", "2024-02-05", 101, 1005, "5", "229", "USD", "Shipped", "field_sales", "REP001", "18.0", "Express"),
        (7011, 1, "2024-02-04", "2024-02-06", 108, 1002, "1", "329.0", "USD", "shipped", "online", "WEB003", "3.5", "Standard"),
    ],
    [
        "orderId",
        "orderLineId", 
        "orderDate",
        "shipDate",
        "customerId",
        "productId",
        "orderedQty",
        "unitPrice",
        "currency",
        "orderStatus",
        "salesChannel",
        "salesRepId",
        "discountPct",
        "shippingMethod",
    ],
)

In [0]:
Customers_write = "/mnt/data/bronze/customers"

(
    bronze_customers_initial_df
    .write
    .format("delta")
    .mode("overwrite")   # Bronze is overwritten with raw ingestion
    .option("overwriteSchema", "true")
    .save(Customers_write)
)


In [0]:
products_write = "/mnt/data/bronze/products"

(
    bronze_products_initial_df
    .write
    .format("delta")
    .mode("overwrite")   # Bronze is overwritten with raw ingestion
    .option("overwriteSchema", "true")
    .save(products_write)
)

In [0]:
orders_write = "/mnt/data/bronze/orders"

(
    bronze_orders_initial_df
    .write
    .format("delta")
    .mode("overwrite")   # Bronze is overwritten with raw ingestion
    .option("overwriteSchema", "true")
    .save(orders_write)
)